In [ ]:
# 1. 라이브러리 및 경로 설정
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np
from collections import defaultdict

ROOT = Path("/data1/workspaces/tp2026/scooter-anomaly-detection")
DATA_ROOT = ROOT / "data/yolo"
RESULT_ROOT = ROOT / "results"

# 0: background, 1: normal, 2: abnormal
NUM_CLASSES = 3
CLASS_NAMES = {1: "normal", 2: "abnormal"}

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

In [ ]:
# 2. Dataset 클래스 정의
# YOLO txt 라벨 → Faster R-CNN 형식으로 변환
# YOLO: class cx cy w h (정규화)
# Faster R-CNN: [x1, y1, x2, y2] (픽셀 좌표)

class ScooterDataset(Dataset):
    def __init__(self, split):
        self.img_dir = DATA_ROOT / "images" / split
        self.lbl_dir = DATA_ROOT / "labels" / split
        self.image_paths = sorted(self.img_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        lbl_path = self.lbl_dir / f"{img_path.stem}.txt"

        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        img_tensor = torchvision.transforms.functional.to_tensor(img)

        boxes = []
        labels = []

        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])

                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)  # 0=bg, 1=normal, 2=abnormal

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        return img_tensor, target

In [ ]:
# 3. Dataset 검증

train_dataset = ScooterDataset("train")
val_dataset = ScooterDataset("val")
test_dataset = ScooterDataset("test")

print(f"train: {len(train_dataset)}")
print(f"val:   {len(val_dataset)}")
print(f"test:  {len(test_dataset)}")

img, target = train_dataset[0]
print(f"\n이미지 shape: {img.shape}")
print(f"boxes: {target['boxes'].shape}")
print(f"labels: {target['labels']}")

In [ ]:
# 4. 모델 정의 및 유틸 함수

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def collate_fn(batch):
    return tuple(zip(*batch))

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter + 1e-8
    return inter / union

def compute_ap(preds, targets, iou_threshold):
    tp_list, fp_list, scores_list, n_gt = [], [], [], 0

    for (pred_boxes, pred_scores), gt_boxes in zip(preds, targets):
        n_gt += len(gt_boxes)
        matched = set()
        sorted_idx = np.argsort(-pred_scores) if len(pred_scores) > 0 else []

        for idx in sorted_idx:
            box = pred_boxes[idx]
            best_iou, best_gt = 0, -1

            for j, gt_box in enumerate(gt_boxes):
                iou = compute_iou(box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt = j

            if best_iou >= iou_threshold and best_gt not in matched:
                tp_list.append(1)
                fp_list.append(0)
                matched.add(best_gt)
            else:
                tp_list.append(0)
                fp_list.append(1)
            scores_list.append(pred_scores[idx])

    if n_gt == 0:
        return 0.0

    sorted_idx = np.argsort(-np.array(scores_list)) if scores_list else []
    tp_cum = np.cumsum(np.array(tp_list)[sorted_idx]) if len(tp_list) > 0 else np.array([])
    fp_cum = np.cumsum(np.array(fp_list)[sorted_idx]) if len(fp_list) > 0 else np.array([])

    precision = tp_cum / (tp_cum + fp_cum + 1e-8)
    recall = tp_cum / (n_gt + 1e-8)
    ap = float(np.trapezoid(precision, recall)) if len(precision) > 0 else 0.0
    return ap

def evaluate(model, dataset, device, iou_thresholds=None):
    if iou_thresholds is None:
        iou_thresholds = [0.5]

    model.eval()
    loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

    all_preds = {cls: [] for cls in [1, 2]}
    all_targets = {cls: [] for cls in [1, 2]}
    all_tp, all_fp, all_fn = 0, 0, 0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                for cls in [1, 2]:
                    mask = output["labels"] == cls
                    pred_boxes = output["boxes"][mask].cpu().numpy()
                    pred_scores = output["scores"][mask].cpu().numpy()

                    gt_mask = target["labels"] == cls
                    gt_boxes = target["boxes"][gt_mask].cpu().numpy()

                    all_preds[cls].append((pred_boxes, pred_scores))
                    all_targets[cls].append(gt_boxes)

                # Precision / Recall 계산용
                pred_boxes_all = output["boxes"].cpu().numpy()
                pred_scores_all = output["scores"].cpu().numpy()
                gt_boxes_all = target["boxes"].cpu().numpy()

                matched = set()
                for i, (box, score) in enumerate(zip(pred_boxes_all, pred_scores_all)):
                    if score < 0.5:
                        continue
                    best_iou, best_gt = 0, -1
                    for j, gt_box in enumerate(gt_boxes_all):
                        iou = compute_iou(box, gt_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt = j
                    if best_iou >= 0.5 and best_gt not in matched:
                        all_tp += 1
                        matched.add(best_gt)
                    else:
                        all_fp += 1
                all_fn += len(gt_boxes_all) - len(matched)

    # mAP50
    ap50_per_class = {cls: compute_ap(all_preds[cls], all_targets[cls], 0.5) for cls in [1, 2]}
    map50 = float(np.mean(list(ap50_per_class.values())))

    # mAP50-95
    iou_range = np.arange(0.5, 1.0, 0.05)
    ap_per_iou = []
    for iou_t in iou_range:
        ap_cls = [compute_ap(all_preds[cls], all_targets[cls], iou_t) for cls in [1, 2]]
        ap_per_iou.append(np.mean(ap_cls))
    map50_95 = float(np.mean(ap_per_iou))

    # Precision / Recall / F1
    precision = all_tp / (all_tp + all_fp + 1e-8)
    recall = all_tp / (all_tp + all_fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        "mAP50": round(map50, 3),
        "mAP50-95": round(map50_95, 3),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "normal_mAP50": round(ap50_per_class[1], 3),
        "abnormal_mAP50": round(ap50_per_class[2], 3),
        "normal_mAP50-95": round(float(np.mean([compute_ap(all_preds[1], all_targets[1], t) for t in iou_range])), 3),
        "abnormal_mAP50-95": round(float(np.mean([compute_ap(all_preds[2], all_targets[2], t) for t in iou_range])), 3),
    }

In [ ]:
# 5. 학습 및 평가 함수

def run_faster_rcnn(epochs, exp_name):
    print(f"\n{'='*40}")
    print(f"실험 시작: {exp_name}")
    print(f"{'='*40}")

    train_dataset = ScooterDataset("train")
    val_dataset = ScooterDataset("val")
    test_dataset = ScooterDataset("test")

    train_loader = DataLoader(
        train_dataset, batch_size=4, shuffle=True,
        collate_fn=collate_fn, num_workers=2
    )

    model = get_model(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    save_dir = RESULT_ROOT / f"{exp_name}_train"
    save_dir.mkdir(parents=True, exist_ok=True)
    best_map = 0
    best_model_path = save_dir / "best.pth"

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        val_result = evaluate(model, val_dataset, DEVICE)
        print(f"Epoch {epoch}/{epochs} | loss: {total_loss/len(train_loader):.4f} | val_mAP50: {val_result['mAP50']:.3f} | val_mAP50-95: {val_result['mAP50-95']:.3f}")

        if val_result["mAP50"] > best_map:
            best_map = val_result["mAP50"]
            torch.save(model.state_dict(), best_model_path)

    print(f"\nbest val_mAP50: {best_map:.3f}")
    print("test 평가 중...")

    model.load_state_dict(torch.load(best_model_path))
    test_result = evaluate(model, test_dataset, DEVICE)
    test_result["name"] = exp_name

    print(f"\n===== {exp_name} test 결과 =====")
    print(f"{'모델':<25} {'mAP50':>8} {'mAP50-95':>10} {'P':>8} {'R':>8} {'F1':>6} {'normal_50':>10} {'abnormal_50':>12} {'normal_95':>10} {'abnormal_95':>12}")
    print("-" * 110)
    print(f"{test_result['name']:<25} {test_result['mAP50']:>8} {test_result['mAP50-95']:>10} {test_result['precision']:>8} {test_result['recall']:>8} {test_result['f1']:>6} {test_result['normal_mAP50']:>10} {test_result['abnormal_mAP50']:>12} {test_result['normal_mAP50-95']:>10} {test_result['abnormal_mAP50-95']:>12}")

    return test_result

In [ ]:
# 6. 실험 실행 — epoch 20, 25, 30

results_frcnn = []

for epochs in [20, 25, 30]:
    exp_name = f"faster_rcnn_ep{epochs}"
    result = run_faster_rcnn(epochs, exp_name)
    results_frcnn.append(result)

In [ ]:
# 7. 전체 결과 비교

print(f"\n===== Faster R-CNN 전체 결과 =====")
print(f"{'모델':<25} {'mAP50':>8} {'mAP50-95':>10} {'P':>8} {'R':>8} {'F1':>6} {'normal_50':>10} {'abnormal_50':>12} {'normal_95':>10} {'abnormal_95':>12}")
print("-" * 110)
for r in results_frcnn:
    print(f"{r['name']:<25} {r['mAP50']:>8} {r['mAP50-95']:>10} {r['precision']:>8} {r['recall']:>8} {r['f1']:>6} {r['normal_mAP50']:>10} {r['abnormal_mAP50']:>12} {r['normal_mAP50-95']:>10} {r['abnormal_mAP50-95']:>12}")